### Setup

make sure FastAPI server is running in terminal `fastapi dev main.py`

In [5]:
import httpx
import time
import pandas as pd

API_URL = "http://127.0.0.1:8000/enrich"
print("Helper functions loaded. API Target:", API_URL)

Helper functions loaded. API Target: http://127.0.0.1:8000/enrich


### Test 1 - Full Pipeline (HTTP POST)

In [6]:
payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC01514953",
        "identifierType": "ac",
        "gndId": "115680667",
        "wikidataId": "Q61915",
        "isbn": "3446144900",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": True,
        "fetchSimilarByAuthor": True,
        "fetchSimilarBySubject": True,
        "fetchSimilarByClassification": True,
        "fetchLobidGND": True,
        "fetchWikidata": True,
        "fetchCover": True,
        "fetchDescription": True,
        "maxRecs": 5
    }
}

t0 = time.perf_counter()
response = httpx.post(API_URL, json=payload, timeout=15.0)
elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)

print(f"Status Code: {response.status_code} ({elapsed_ms}ms)")
assert response.status_code == 200, "API returned non-200 status code"

data = response.json()
result = data["response"]["result"]

print("Full Pipeline HTTP Test Passed!")
print("   - Title:", result["basicMarc21MD"]["title"]["titleMain"])
print("   - GND Name:", result["gndInfoLobid"]["gndInformation"]["preferredName"])
print("   - Wikidata ID:", result["wikidataData"]["wikidataId"])
print("   - Cover URL:", result["bookCover"]["coverURL"])

Status Code: 200 (6781.79ms)
Full Pipeline HTTP Test Passed!
   - Title: Schopenhauer und die wilden Jahre der Philosophie
   - GND Name: Safranski, Rüdiger
   - Wikidata ID: Q61915
   - Cover URL: https://covers.openlibrary.org/b/isbn/3446144900-M.jpg?default=false


### Test 2 — Minimal Fetch (Fast Path)

In [8]:
payload_minimal = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC01514953",
        "identifierType": "ac",
        "fetchMarc21MD": True,
        "fetchSimilarSRU": False,
        "fetchLobidGND": False,
        "fetchWikidata": False,
        "fetchCover": False,
        "fetchDescription": False
    }
}

t0 = time.perf_counter()
response = httpx.post(API_URL, json=payload_minimal, timeout=10.0)
elapsed_ms = round((time.perf_counter() - t0) * 1000, 2)

assert response.status_code == 200
result = response.json()["response"]["result"]

assert result["basicMarc21MD"] is not None
assert result["gndInfoLobid"] is None
assert result["bookCover"] is None

print(f"Minimal Fetch Test Passed in {elapsed_ms}ms!")

Minimal Fetch Test Passed in 695.25ms!


### Test 3 — Direct ISBN Bypass (No MARC21 Lookup)

In [9]:
payload_bypass = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "isbn": "3446144900",
        "fetchMarc21MD": False,
        "fetchCover": True,
        "fetchDescription": True
    }
}

response = httpx.post(API_URL, json=payload_bypass, timeout=10.0)
assert response.status_code == 200
result = response.json()["response"]["result"]

assert result["basicMarc21MD"] is None
assert result["bookCover"] is not None

print("ISBN Bypass Test Passed!")
print("   - Cover URL:", result["bookCover"]["coverURL"])

ISBN Bypass Test Passed!
   - Cover URL: https://covers.openlibrary.org/b/isbn/3446144900-M.jpg?default=false


### Quick Response Time Benchmark Matrix

In [10]:
scenarios = [
    ("Full Pipeline", True, True, True, True, True),
    ("MARC21 Only", True, False, False, False, False),
    ("MARC21 + GND/Wikidata", True, False, True, True, False),
    ("MARC21 + Cover/Desc", True, False, False, False, True),
]

audit_results = []

for name, marc, sru, lobid, wiki, media in scenarios:
    p = {
        "iType": "bib",
        "institution": {
            "iName": "oenb",
            "identifier": "AC01514953",
            "identifierType": "ac",
            "fetchMarc21MD": marc,
            "fetchSimilarSRU": sru,
            "fetchLobidGND": lobid,
            "fetchWikidata": wiki,
            "fetchCover": media,
            "fetchDescription": media
        }
    }
    t0 = time.perf_counter()
    r = httpx.post(API_URL, json=p, timeout=15.0)
    dur = round((time.perf_counter() - t0) * 1000, 2)
    
    audit_results.append({
        "Scenario": name,
        "HTTP Status": r.status_code,
        "Total Time (ms)": dur
    })

pd.DataFrame(audit_results)

,Scenario,HTTP Status,Total Time (ms)
0,Full Pipeline,200,2937.22
1,MARC21 Only,200,294.34
2,MARC21 + GND/Wikidata,200,379.06
3,MARC21 + Cover/Desc,200,2224.47
